In [ ]:
import os
import json
import datetime
from typing import TypedDict, List, Optional, Literal, Annotated
import operator

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
ModelName = "llama-3.3-70b-versatile"
LogDirectory = "research_reports"

AgentSupervisor = "supervisor"
AgentWebResearcher = "web_researcher"
AgentDataAnalyst = "data_analyst"
AgentReportWriter = "report_writer"

if not GroqApiKey:
    raise ValueError("GroqAPI not found in .env")

print(f"Model: {ModelName}")
print(f"Log Dir: {LogDirectory}")

In [ ]:
class SystemState(TypedDict):
    
    Topic: str
    SubTasks: List[dict]
    AgentOutputs: dict
    NextAgent: Optional[str]
    CurrentTaskIndex: int
    FinalReport: str
    DelegationTrace: Annotated[List[dict], operator.add]

In [ ]:
LLM = ChatGroq(api_key=GroqApiKey, model=ModelName, temperature=0.4)
SearchTool = DuckDuckGoSearchRun() 

In [ ]:
def makeTraceEntry(AgentName: str, Action: str, Content: str) -> dict:
    
    return { "timestamp" : datetime.datetime.now().strftime("%H:%M:%S"), "agent": AgentName, "action": Action, "content": Content, }


def printAgentBanner(AgentName: str, TaskDescription: str):
    
    Label = AgentName.replace("_", " ").upper()
    print("\n\n")
    print(f"[{Label}]")
    print(f"Task: {TaskDescription}")

In [ ]:
def supervisorAgent(State: SystemState) -> Command:
   
    Topic = State["Topic"]
    SubTasks = State.get("SubTasks", [])
    CurrentTaskIndex = State.get("CurrentTaskIndex", 0)
    AgentOutputs = State.get("AgentOutputs", {})
    
    if not SubTasks:
        printAgentBanner(AgentSupervisor, f"Planning research on: {Topic}")

        PlanningMessages = [
            SystemMessage(content=(
                "You are a research supervisor. Break the given topic into exactly 3 subtasks. "
                "Assign each to one of these agents: web_researcher, data_analyst, report_writer. "
                "Rules: "
                "  - web_researcher: searches for facts, statistics, background information "
                "  - data_analyst: interprets numbers, trends, comparisons from research findings "
                "  - report_writer: composes the final structured report (always assign last) "
                "Return ONLY a valid JSON array of objects. Each object must have exactly two keys: "
                "  'agent' (one of the three names above) and 'task' (a specific instruction string). "
                "Example: [{\"agent\": \"web_researcher\", \"task\": \"Find key facts about X\"}] "
                "No markdown, no explanation, just the JSON array."
            )),
            HumanMessage(content=f"Research topic: {Topic}")
        ]

        Response = LLM.invoke(PlanningMessages)
        RawPlan = Response.content.strip()

        try:
            CleanPlan = RawPlan.replace("```json", "").replace("```", "").strip()
            SubTasks  = json.loads(CleanPlan)

            SubTasks.sort(key=lambda X: 1 if X["agent"] == "report_writer" else 0)
        except (json.JSONDecodeError, ValueError):

            SubTasks = [
                {"agent": "web_researcher", "task": f"Research key facts about: {Topic}"},
                {"agent": "data_analyst", "task": f"Analyse any data or trends found about: {Topic}"},
                {"agent": "report_writer", "task": f"Write a comprehensive report on: {Topic}"},
            ]

        PlanSummary = "\n".join( [f"  {I+1}. [{T['agent']}] {T['task']}" for I, T in enumerate(SubTasks)] )
        print(f"Plan created ({len(SubTasks)} subtasks):\n{PlanSummary}")

        TraceEntry   = makeTraceEntry(AgentSupervisor, "Plan", PlanSummary)
        CurrentTaskIndex = 0

    else:
        TraceEntry = makeTraceEntry( AgentSupervisor, "Route", f"Worker finished. Checking next task (index {CurrentTaskIndex})." )

    if CurrentTaskIndex < len(SubTasks):
        NextTask = SubTasks[CurrentTaskIndex]
        NextAgent = NextTask["agent"]

        print(f"\n[Supervisor]: Delegating to [{NextAgent}] (task {CurrentTaskIndex + 1}/{len(SubTasks)})")

        return Command( goto = NextAgent, update = { "SubTasks": SubTasks, "CurrentTaskIndex": CurrentTaskIndex, "NextAgent": NextAgent, "DelegationTrace": [TraceEntry], } )

    else:
        print(f"\n[Supervisor] All subtasks complete. Routing to report_writer.")

        return Command( goto = AgentReportWriter, update = { "SubTasks": SubTasks, "CurrentTaskIndex": CurrentTaskIndex, "DelegationTrace": [TraceEntry], } )

In [ ]:
def webResearcherAgent(State: SystemState) -> Command:

    SubTasks = State.get("SubTasks", [])
    CurrentTaskIndex = State.get("CurrentTaskIndex", 0)
    Topic = State["Topic"]
    AgentOutputs = State.get("AgentOutputs", {}).copy()

    MyTask = SubTasks[CurrentTaskIndex]["task"] if CurrentTaskIndex < len(SubTasks) else Topic

    printAgentBanner(AgentWebResearcher, MyTask)

    print("\tSearching the web")
    try:
        SearchQuery = f"{Topic} {MyTask[:50]}"
        SearchResults = SearchTool.run(SearchQuery)
        print(f"  Search returned {len(SearchResults)} characters.")
        
    except Exception as SearchError:
        SearchResults = f"Search unavailable: {SearchError}. Using general knowledge."
        print(f"  Search failed: {SearchError}")

    SynthesisMessages = [
        SystemMessage(content=(
            "You are a web researcher. You have been given search results and a research task."
            "Extract and summarise the most relevant facts, statistics, and key information. "
            "Be specific and factual. Structure your output clearly. Aim for 200-300 words."
        )),
        HumanMessage(content=(
            f"Research task: {MyTask}\n\n"
            f"Search results:\n{SearchResults[:3000]}\n\n"
            "Please extract and summarise the key findings."
        ))
    ]

    Response = LLM.invoke(SynthesisMessages)
    ResearchOutput = Response.content

    print(f"\nOutput:\n{ResearchOutput[:300]}" if len(ResearchOutput) > 300 else f"\nOutput:\n{ResearchOutput}")

    AgentOutputs[AgentWebResearcher] = ResearchOutput
    TraceEntry = makeTraceEntry(AgentWebResearcher, "Research", ResearchOutput)

    return Command( goto = AgentSupervisor, update = { "AgentOutputs": AgentOutputs, "CurrentTaskIndex": CurrentTaskIndex + 1, "DelegationTrace": [TraceEntry], } )

In [ ]:
def dataAnalystAgent(State: SystemState) -> Command:
    
    SubTasks = State.get("SubTasks", [])
    CurrentTaskIndex = State.get("CurrentTaskIndex", 0)
    Topic = State["Topic"]
    AgentOutputs = State.get("AgentOutputs", {}).copy()

    MyTask = SubTasks[CurrentTaskIndex]["task"] if CurrentTaskIndex < len(SubTasks) else f"Analyse data about {Topic}"
    ResearchInput = AgentOutputs.get(AgentWebResearcher, "No prior research available.")

    printAgentBanner(AgentDataAnalyst, MyTask)

    AnalysisMessages = [
        SystemMessage(content=(
            "You are a data analyst. You receive research findings and produce analytical insights. "
            "Focus on: numerical data and statistics, trends and patterns, comparisons and rankings, "
            "key metrics, growth rates, and any quantitative observations. "
            "If no explicit data is present, analyse qualitative trends analytically. "
            "Structure your analysis clearly with headers. Aim for 200-250 words."
        )),
        HumanMessage(content=(
            f"Analysis task: {MyTask}\n\n"
            f"Research findings to analyse:\n{ResearchInput}\n\n"
            "Provide a thorough data analysis."
        ))
    ]

    Response = LLM.invoke(AnalysisMessages)
    AnalysisOutput = Response.content

    print(f"\nOutput:\n{AnalysisOutput[:300]}" if len(AnalysisOutput) > 300 else f"\nOutput:\n{AnalysisOutput}")

    AgentOutputs[AgentDataAnalyst] = AnalysisOutput
    TraceEntry = makeTraceEntry(AgentDataAnalyst, "Analysis", AnalysisOutput)

    return Command( goto = AgentSupervisor, update = { "AgentOutputs": AgentOutputs, "CurrentTaskIndex": CurrentTaskIndex + 1, "DelegationTrace": [TraceEntry], } )

In [ ]:
def reportWriterAgent(State: SystemState) -> Command:

    Topic = State["Topic"]
    AgentOutputs = State.get("AgentOutputs", {})

    ResearchSection = AgentOutputs.get(AgentWebResearcher, "No research provided.")
    AnalysisSection = AgentOutputs.get(AgentDataAnalyst, "No analysis provided.")

    printAgentBanner(AgentReportWriter, f"Composing final report on: {Topic}")

    ReportMessages = [
        SystemMessage(content=(
            "You are a professional report writer. "
            "You receive research findings and data analysis, then compose a comprehensive report. "
            "Structure the report with the following sections:\n"
            "1. Executive Summary (2-3 sentences)\n"
            "2. Key Findings (from research)\n"
            "3. Data & Analysis\n"
            "4. Conclusions and Implications\n"
            "Use clear headings. Professional tone. Aim for 400-500 words total."
        )),
        HumanMessage(content=(
            f"Report topic: {Topic}\n\n"
            f"\tResearch Findings\n{ResearchSection}\n\n"
            f"\tData Analysis\n{AnalysisSection}\n\n"
            "Please compose the final report."
        ))
    ]

    Response = LLM.invoke(ReportMessages)
    FinalReport = Response.content

    print(f"\nReport (first 400 characters):\n{FinalReport[:400]}")

    TraceEntry = makeTraceEntry(AgentReportWriter, "Report", FinalReport)

    return Command( goto = END, update = { "FinalReport": FinalReport, "DelegationTrace": [TraceEntry], } )

In [ ]:
def buildMultiAgentGraph():
    
    Builder = StateGraph(SystemState)
    Memory = MemorySaver()

    Builder.add_node(AgentSupervisor, supervisorAgent)
    Builder.add_node(AgentWebResearcher, webResearcherAgent)
    Builder.add_node(AgentDataAnalyst, dataAnalystAgent)
    Builder.add_node(AgentReportWriter, reportWriterAgent)

    Builder.add_edge(START, AgentSupervisor)

    Graph = Builder.compile(checkpointer=Memory)

    return Graph

MultiAgentGraph = buildMultiAgentGraph()

In [ ]:
def saveReportAndTrace(Topic: str, FinalReport: str, DelegationTrace: List[dict]):
    
    os.makedirs(LogDirectory, exist_ok=True)
    Timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    SafeTopic = Topic[:40].replace(" ", "_").replace("/", "-")
    LogFilePath = os.path.join(LogDirectory, f"report_{SafeTopic}_{Timestamp}.txt")

    with open(LogFilePath, "w", encoding="utf-8") as F:
        
        F.write("\n")
        F.write("Multi Agent Report\n")
        F.write(f"Topic: {Topic}\n")
        F.write(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        F.write(f"Model: {ModelName}\n")
        F.write("\n\n")

        F.write("Delegation Trace\n")
        F.write("\n")
        
        for Entry in DelegationTrace:
            F.write(f"[{Entry['timestamp']}] [{Entry['agent'].upper()}] {Entry['action']}\n")

            ContentPreview = Entry['content'][:500]
            if len(Entry['content']) > 500:
                ContentPreview += "[truncated]"
            F.write(f"{ContentPreview}\n")
            F.write("\n")

        F.write("\n\n")
        F.write("Final Report\n")
        F.write("\n\n")
        
        F.write(FinalReport)
        F.write("\n")

    print(f"\nSaved to: {LogFilePath}")
    return LogFilePath

In [ ]:
def runResearch(Topic: str) -> dict:
    
    print("\n")
    print("  Multi Agent Research and Report System")
    print(f"Topic: {Topic}")

    Timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    ThreadId = f"research_{Timestamp}"
    Config = {"configurable": {"thread_id": ThreadId}}

    InitialState = { "Topic": Topic, "SubTasks": [], "AgentOutputs": {}, "NextAgent": None, "CurrentTaskIndex": 0, "FinalReport": "", "DelegationTrace": [], }

    FinalState = MultiAgentGraph.invoke(InitialState, config=Config)

    FinalReport = FinalState.get("FinalReport", "")
    DelegationTrace = FinalState.get("DelegationTrace", [])

    print("\nFinal Report\n")
    print(FinalReport)

    if FinalReport:
        saveReportAndTrace(Topic, FinalReport, DelegationTrace)
    else:
        print("[Warning] No final report was generated.")

    return FinalState

In [ ]:
ResearchTopic = "The current state and growth trends of the global electric vehicle market"
Result = runResearch(ResearchTopic)

In [ ]:
def runInteractiveResearch():
    
    print("\nMulti-Agent Research System, Interactive")
    print("Type a research topic, or 'quit' to exit\n")

    while True:
        try:
            Topic = input("\nResearch topic: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if Topic.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break

        if not Topic:
            print("(Please enter a topic.)")
            continue

        try:
            runResearch(Topic)
        except Exception as Error:
            print(f"\n[Error]: {Error}")
            print("Please try a different topic.")

runInteractiveResearch()

In [ ]:
def printDelegationTrace(FinalState: dict):
    
    Trace = FinalState.get("DelegationTrace", [])

    print("\n Delegation Trace")
    print(f"({len(Trace)} entries)\n")

    for Index, Entry in enumerate(Trace, start=1):
        print(f"\n  Step {Index}: [{Entry['agent'].upper()}] {Entry['action']} @ {Entry['timestamp']}")

        ContentPreview = Entry['content'][:200]
        if len(Entry['content']) > 200:
            ContentPreview += ".."
        print(f"  {ContentPreview}")

    print("\n")


if 'Result' in dir():
    printDelegationTrace(Result)
else:
    print("Run Cell 14 first to generate a result.")